<a href="https://colab.research.google.com/github/noviantisafitri/LINE-App-Review-Sentiment-Analysis/blob/main/Sentimen_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Analisis Sentimen Review Aplikasi LINE di Play Store**  


Notebook ini berisi proses **Analisis Sentimen** terhadap **30.000 data review aplikasi LINE** di Play Store menggunakan teknik **Machine Learning dan Deep Learning**. Tujuan dari analisis ini adalah untuk mengklasifikasikan review ke dalam **3 kategori sentimen**: **positif, netral, dan negatif**.  

Dalam analisis ini, dilakukan **3 skema pelatihan** dengan kombinasi algoritma, metode ekstraksi fitur, dan pembagian data yang berbeda.  

---

## **Langkah-langkah yang Dilakukan**  

#### **1. Menyiapkan Data**  
- Dataset terdiri dari **30.000 data review aplikasi LINE**.  
- Data dikategorikan ke dalam **3 kelas**: **positif, netral, dan negatif**.  
- Dilakukan **pembersihan data**, termasuk:
  - Menghapus nilai kosong.
  - Mengonversi teks ke huruf kecil.
  - Mendeteksi kata-kata negatif dan positif untuk membantu proses pelabelan.  

---

#### **2. Melakukan 3 Skema Pelatihan Berbeda**  

##### **Skema 1: CNN + Tokenisasi & Padding (80/20)**
- **Model**: **Convolutional Neural Network (CNN)**  
- **Ekstraksi Fitur**: Tokenisasi dan Padding  
- **Pembagian Data**: **80% training, 20% testing**  
- **Penjelasan**:  
  - Data teks dikonversi menjadi **sekuens angka** menggunakan **Tokenisasi**.  
  - Panjang teks diseragamkan dengan **Padding**.  
  - Model CNN digunakan untuk menangkap pola fitur dalam teks menggunakan **Conv1D**, **BatchNormalization**, dan **GlobalMaxPooling1D**.  

---

##### **Skema 2: SVM + TF-IDF (70/30)**
- **Model**: **Support Vector Machine (SVM) dengan kernel linear**  
- **Ekstraksi Fitur**: **TF-IDF (Term Frequency - Inverse Document Frequency)**  
- **Pembagian Data**: **70% training, 30% testing**  
- **Penjelasan**:  
  - Data teks dikonversi menjadi vektor numerik menggunakan **TF-IDF**.  
  - Model **SVM (Support Vector Machine)** digunakan untuk klasifikasi berbasis **margin maksimal**, yang bekerja dengan baik pada data berdimensi tinggi seperti teks.  

---

##### **Skema 3: Random Forest + TF-IDF (80/20)**
- **Model**: **Random Forest Classifier**  
- **Ekstraksi Fitur**: **TF-IDF**  
- **Pembagian Data**: **80% training, 20% testing**  
- **Penjelasan**:  
  - Seperti pada Skema 2, data diubah menjadi vektor numerik menggunakan **TF-IDF**.  
  - Model **Random Forest** dengan **100 pohon keputusan** digunakan sebagai pendekatan **ensemble learning**, di mana prediksi akhir diperoleh dari voting mayoritas pohon-pohon dalam model.  

---

#### **3. Evaluasi Model**  
- Setiap model dievaluasi berdasarkan **akurasi pada training set dan testing set**.  
- **Target akurasi** yang diinginkan adalah **≥ 92%** pada training dan testing set.  
- Jika model gagal mencapai target, maka dilakukan **penyesuaian parameter atau metode ekstraksi fitur**.  

---

#### **4. Testing & Inference**  
- Setelah pelatihan selesai, model diuji dengan **beberapa review baru** untuk melihat apakah prediksi sentimen sesuai dengan ekspektasi.  
- **Hasil inference** akan berupa **kategori sentimen** (**negatif, netral, atau positif**), sehingga bisa digunakan untuk menganalisis kepuasan pengguna terhadap aplikasi LINE.  

---

## Library Import yang digunakan

In [37]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
import tensorflow as tf
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

## Data Processing

In [38]:
df_reviews = pd.read_csv('https://raw.githubusercontent.com/noviantisafitri/LINE-App-Review-Sentiment-Analysis/refs/heads/main/Data/line_reviews.csv')
df_reviews.head()

,review,rating,date,username
0,sangat kecewa karena fitur story line di hapus...,5,2025-02-14 12:53:36,Pengguna Google
1,Saya sangat kecewa atas notifikasi Line VOOM y...,5,2025-01-23 17:22:34,Pengguna Google
2,Kecewa banget line voom hilang di Indonesia pa...,5,2025-02-14 07:12:31,Pengguna Google
3,Tolong perbaiki masalah ketika login pake apli...,5,2025-01-03 16:12:50,Pengguna Google
4,line jangan di hapus apapun dong apalagi fitur...,5,2025-01-11 12:48:28,Pengguna Google


- review → Berisi teks ulasan pengguna terhadap aplikasi LINE. Contohnya, ada pengguna yang kecewa dengan fitur yang dihapus.
- rating → Skor yang diberikan pengguna untuk aplikasi LINE (dalam skala 1-5). Semua review dalam contoh ini memiliki rating 5.
- date → Tanggal dan waktu ketika review diberikan oleh pengguna. Formatnya adalah YYYY-MM-DD HH:MM:SS.
- username → Nama pengguna yang memberikan review. Di sini, semua review diberikan oleh "Pengguna Google", menunjukkan bahwa nama pengguna tidak selalu ditampilkan secara eksplisit.

### Menghapus nilai kosong

In [39]:
# Mengganti NaN dengan string kosong dan pastikan bertipe string
df_reviews['review'] = df_reviews['review'].fillna('').astype(str)

### Melakukan Labeling Data

In [40]:
# Daftar keywords
negative_keywords = [
    'kecewa', 'buruk', 'jelek', 'tidak bagus', 'parah', 'error', 'masalah',
    'benci', 'sulit', 'gangguan', 'lemot', 'kurang', 'mengecewakan', 'payah',
    'tidak puas', 'hilang', 'hapus', 'crash', 'lag', 'bug'
]

positive_keywords = [
    'bagus', 'baik', 'puas', 'senang', 'suka', 'cinta', 'keren', 'hebat',
    'luar biasa', 'mantap', 'memuaskan', 'mudah', 'cepat', 'membantu',
    'nyaman', 'sempurna', 'rekomendasi', 'terbaik', 'sangat suka', 'cocok'
]

In [41]:
# Fungsi untuk deteksi kata
def contains_negative_words(text):
    text = text.lower()
    return any(re.search(r'\b' + re.escape(word) + r'\b', text) for word in negative_keywords)

def contains_positive_words(text):
    text = text.lower()
    return any(re.search(r'\b' + re.escape(word) + r'\b', text) for word in positive_keywords)

In [42]:
# Fungsi untuk labeling
def sentiment_label(row):
    rating = row['rating']
    review = row['review']
    if rating == 5 and contains_negative_words(review):
        return 'negative'
    elif rating in [4, 5] and contains_positive_words(review):
        return 'positive'
    elif rating == 1 or contains_negative_words(review):
        return 'negative'
    else:
        return 'neutral'

In [43]:
# Proses labeling
df_reviews['sentiment'] = df_reviews.apply(sentiment_label, axis=1)
label_encoder = LabelEncoder()
df_reviews['sentiment_encoded'] = label_encoder.fit_transform(df_reviews['sentiment'])

print("\nContoh data setelah labeling:")
df_reviews[['review', 'rating', 'sentiment', 'sentiment_encoded']].head()


Contoh data setelah labeling:


,review,rating,sentiment,sentiment_encoded
0,sangat kecewa karena fitur story line di hapus...,5,negative,0
1,Saya sangat kecewa atas notifikasi Line VOOM y...,5,negative,0
2,Kecewa banget line voom hilang di Indonesia pa...,5,negative,0
3,Tolong perbaiki masalah ketika login pake apli...,5,negative,0
4,line jangan di hapus apapun dong apalagi fitur...,5,negative,0


1. **review** → Ulasan pengguna tentang aplikasi LINE.  
2. **rating** → Skor yang diberikan pengguna (dalam skala 1-5).  
3. **sentiment** → Hasil labeling, yang menunjukkan apakah ulasan bersifat **positif, netral, atau negatif**. Pada contoh ini, semua review diberi label **negative**.  
4. **sentiment_encoded** → Representasi numerik dari sentiment:  
   - 0 → **Negative**  
   - 1 → **Neutral**  
   - 2 → **Positive**  


In [44]:
print("\nDistribusi label sentimen:")
df_reviews['sentiment'].value_counts()


Distribusi label sentimen:


,count
sentiment,
neutral,17681
positive,11334
negative,985


Mayoritas review memiliki sentimen netral (58.94%), yang berarti banyak pengguna memberikan ulasan yang tidak terlalu condong ke positif atau negatif.
Sentimen positif (37.78%) cukup tinggi, menunjukkan banyak pengguna yang puas dengan aplikasi LINE. Sentimen negatif hanya 3.28%, yang berarti keluhan atau ketidakpuasan relatif sedikit dibandingkan total review.

### Tokenisasi dan Pembagian Data

In [45]:
# Proses Tokenisasi dan Pembagian data
X = df_reviews['review'].values
y = df_reviews['sentiment_encoded'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

tokenizer = Tokenizer(num_words=10000, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

max_length = 100
X_train_pad = pad_sequences(X_train_seq, padding='post', maxlen=max_length)
X_test_pad = pad_sequences(X_test_seq, padding='post', maxlen=max_length)

print("Shape X_train_pad:", X_train_pad.shape)
print("Shape X_test_pad:", X_test_pad.shape)

Shape X_train_pad: (24000, 100)
Shape X_test_pad: (6000, 100)


Dataset dibagi menjadi **80% (24.000 data) untuk training** dan **20% (6.000 data) untuk testing**. Ulasan dikonversi menjadi angka menggunakan tokenisasi dengan **10.000 kata paling sering muncul**, serta kata yang tidak dikenal diganti dengan `<OOV>`. Semua review dipadatkan menjadi panjang **maksimal 100 kata** menggunakan **padding post**, memastikan ukuran input seragam.

In [46]:
# TF-IDF Ekstraksi Fitur
tfidf = TfidfVectorizer(max_features=10000)
X_train_tfidf = tfidf.fit_transform(X_train).toarray()
X_test_tfidf = tfidf.transform(X_test).toarray()

## Melakukan 3 Skema Pelatihan Berbeda

### Skema 1 - Random Forest dengan Ekstraksi Fitur TF-IDF

Di percobaan ketiga, kita menggunakan model Random Forest Classifier dengan 100 estimators (n_estimators=100).
Ekstraksi fitur juga dilakukan menggunakan TF-IDF untuk mengubah teks menjadi fitur numerik.
Pembagian data menggunakan 80/20 (80% data latih, 20% data uji).

In [47]:
# Inisialisasi dan pelatihan model Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_tfidf, y_train)

# Evaluasi akurasi pada training set dan testing set
train_accuracy_rf = rf_model.score(X_train_tfidf, y_train)
test_accuracy_rf = rf_model.score(X_test_tfidf, y_test)

# Prediksi pada testing set
y_pred_rf = rf_model.predict(X_test_tfidf)

# Menampilkan hasil akurasi
print(f'Training Accuracy Random Forest: {train_accuracy_rf * 100:.2f}%')
print(f'Test Accuracy Random Forest: {test_accuracy_rf * 100:.2f}%')

# Menampilkan classification report
print("\n=== Classification Report Random Forest ===")
print(classification_report(y_test, y_pred_rf, target_names=['negative', 'neutral', 'positive']))

Training Accuracy Random Forest: 100.00%
Test Accuracy Random Forest: 98.50%

=== Classification Report Random Forest ===
              precision    recall  f1-score   support

    negative       1.00      0.71      0.83       208
     neutral       0.99      1.00      0.99      3558
    positive       0.98      0.99      0.99      2234

    accuracy                           0.98      6000
   macro avg       0.99      0.90      0.94      6000
weighted avg       0.99      0.98      0.98      6000



Model **Random Forest** dilatih menggunakan **100 pohon keputusan** dengan **ekstraksi fitur TF-IDF**. Akurasi pada **training set mencapai 100%**, menunjukkan model sangat cocok dengan data latih. Sementara itu, akurasi pada **testing set sebesar 98.50%**, yang menunjukkan model masih berkinerja sangat baik pada data yang belum pernah dilihat.

### Skema 2 - Support Vector Machine (SVM) dengan Ekstraksi Fitur TF-IDF

Pada percobaan kedua, model yang digunakan adalah Support Vector Machine (SVM) dengan kernel linear.
Ekstraksi fitur dilakukan menggunakan TF-IDF (Term Frequency - Inverse Document Frequency) untuk mengubah teks menjadi representasi numerik.
Pembagian data menggunakan 70/30 (70% data latih, 30% data uji).


In [48]:
# Pembagian data menjadi data latih (70%) dan data uji (30%) khusus untuk SVM
X_train_svm, X_test_svm, y_train_svm, y_test_svm = train_test_split(X, y, test_size=0.3, random_state=42)

# Ekstraksi Fitur
tfidf_svm = TfidfVectorizer(max_features=10000)
X_train_tfidf_svm = tfidf_svm.fit_transform(X_train_svm).toarray()
X_test_tfidf_svm = tfidf_svm.transform(X_test_svm).toarray()

# Inisialisasi dan pelatihan model SVM
svm_model = SVC(kernel='linear', random_state=42)
svm_model.fit(X_train_tfidf_svm, y_train_svm)

# Prediksi pada training set dan testing set
y_train_pred_svm = svm_model.predict(X_train_tfidf_svm)
y_test_pred_svm = svm_model.predict(X_test_tfidf_svm)

# Evaluasi akurasi pada training set dan testing set
train_accuracy_svm = accuracy_score(y_train_svm, y_train_pred_svm)
test_accuracy_svm = accuracy_score(y_test_svm, y_test_pred_svm)

print(f'Training Accuracy SVM (70/30): {train_accuracy_svm * 100:.2f}%')
print(f'Test Accuracy SVM (70/30): {test_accuracy_svm * 100:.2f}%')

# Menampilkan classification report
print("\n=== Classification Report SVM ===")
print(classification_report(y_test_svm, y_test_pred_svm, target_names=['negative', 'neutral', 'positive']))

Training Accuracy SVM (70/30): 99.88%
Test Accuracy SVM (70/30): 99.39%

=== Classification Report SVM ===
              precision    recall  f1-score   support

    negative       1.00      0.89      0.94       317
     neutral       0.99      1.00      1.00      5283
    positive       1.00      0.99      0.99      3400

    accuracy                           0.99      9000
   macro avg       1.00      0.96      0.98      9000
weighted avg       0.99      0.99      0.99      9000



Model **Support Vector Machine (SVM)** dilatih menggunakan **ekstraksi fitur TF-IDF** dengan **pembagian data 70% untuk training dan 30% untuk testing**. Akurasi pada **training set mencapai 99.88%**, menunjukkan model sangat baik dalam mengenali pola pada data latih. Akurasi pada **testing set sebesar 99.39%**, menunjukkan performa model yang **stabil dan generalisasi yang sangat baik** pada data yang belum pernah dilihat.

### Skema 3 - Convolutional Neural Network (CNN) dengan Tokenisasi dan Padding

Di percobaan pertama ini, kita akan menggunakan model Convolutional Neural Network (CNN) dengan ekstraksi fitur tokenisasi dan padding. CNN digunakan untuk menangkap pola dalam data teks dengan Conv1D, BatchNormalization, dan GlobalMaxPooling1D.
Pembagian data menggunakan 80/20 (80% data latih, 20% data uji).

In [49]:
vocab_size = 10000
embedding_dim = 100

callbacks = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6)
]

model_cnn = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_length),
    Conv1D(128, 5, activation='relu'),
    BatchNormalization(),
    GlobalMaxPooling1D(),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(3, activation='softmax')  # 3 kelas: positif, netral, negatif
])

model_cnn.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

print("\n=== Training CNN Model ===")
history_cnn = model_cnn.fit(X_train_pad, y_train, epochs=5, batch_size=32, validation_data=(X_test_pad, y_test), callbacks=callbacks)

# Evaluasi akurasi pada training set
train_accuracy_cnn = history_cnn.history['accuracy'][-1]  # Akurasi dari epoch terakhir pada training
val_accuracy_cnn = history_cnn.history['val_accuracy'][-1]  # Akurasi dari epoch terakhir pada validation/testing

print(f'Training Accuracy CNN: {train_accuracy_cnn * 100:.2f}%')
print(f'Test Accuracy CNN: {val_accuracy_cnn * 100:.2f}%')

# Evaluasi model CNN
loss_cnn, accuracy_cnn = model_cnn.evaluate(X_test_pad, y_test)
print(f"CNN Accuracy: {accuracy_cnn * 100:.2f}%")


=== Training CNN Model ===
Epoch 1/5


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


750/750 ━━━━━━━━━━━━━━━━━━━━ 34s 42ms/step - accuracy: 0.8850 - loss: 0.3121 - val_accuracy: 0.9963 - val_loss: 0.0218 - learning_rate: 0.0010
Epoch 2/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 45s 47ms/step - accuracy: 0.9966 - loss: 0.0165 - val_accuracy: 0.9908 - val_loss: 0.0528 - learning_rate: 0.0010
Epoch 3/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 38s 43ms/step - accuracy: 0.9990 - loss: 0.0054 - val_accuracy: 0.9955 - val_loss: 0.0246 - learning_rate: 0.0010
Epoch 4/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 40s 42ms/step - accuracy: 0.9988 - loss: 0.0071 - val_accuracy: 0.9962 - val_loss: 0.0281 - learning_rate: 5.0000e-04
Training Accuracy CNN: 99.88%
Test Accuracy CNN: 99.62%
188/188 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.9967 - loss: 0.0205
CNN Accuracy: 99.63%


Model **Convolutional Neural Network (CNN)** dilatih dengan **embedding layer dan Conv1D** untuk klasifikasi sentimen dengan **3 kelas (positif, netral, negatif)**. Akurasi training mencapai **99.92%**, sedangkan akurasi validasi/testing mencapai **99.62%**, menunjukkan model mampu mengenali pola dengan sangat baik. Evaluasi akhir menghasilkan **CNN Accuracy sebesar 99.58%**, yang menunjukkan **generalization yang baik** dengan performa **hampir sempurna**.

## Tahap Inference

In [50]:
def predict_sentiment(review, model, model_type):
    """
    Melakukan prediksi sentimen terhadap sebuah review dengan model tertentu.

    Parameters:
    - review (str): Teks review yang akan diuji.
    - model: Model yang digunakan (SVM, RF, atau CNN).
    - model_type (str): Jenis model ("SVM", "RF", atau "CNN").

    Returns:
    - Sentimen dalam bentuk kategorikal ("negative", "neutral", "positive").
    """
    # Preprocessing review
    review = [review]  # Ubah menjadi list

    if model_type in ["SVM", "RF"]:
        # Gunakan TF-IDF untuk model berbasis ML (SVM & RF)
        review_tfidf = tfidf.transform(review).toarray()
        prediction = model.predict(review_tfidf)[0]

    elif model_type == "CNN":
        # Gunakan Tokenizer dan padding untuk model CNN
        review_seq = tokenizer.texts_to_sequences(review)
        review_pad = pad_sequences(review_seq, maxlen=max_length, padding='post')
        prediction = np.argmax(model.predict(review_pad), axis=1)[0]

    # Konversi label numerik ke kategorikal
    sentiment_label = label_encoder.inverse_transform([prediction])[0]
    return sentiment_label

In [51]:
# Percobaan testing
test_reviews = [
    "Aplikasi ini sangat bagus, saya sangat puas!",
    "Sangat mengecewakan, banyak bug dan sering crash.",
    "Aplikasi ini lumayan, tidak buruk tapi juga tidak terlalu bagus.",
]

print("\n=== HASIL PREDIKSI SENTIMEN ===")

for review in test_reviews:
    sentiment_svm = predict_sentiment(review, svm_model, "SVM")
    sentiment_rf = predict_sentiment(review, rf_model, "RF")
    sentiment_cnn = predict_sentiment(review, model_cnn, "CNN")

    print(f"\nReview: {review}")
    print(f"- Prediksi SVM: {sentiment_svm}")
    print(f"- Prediksi Random Forest: {sentiment_rf}")
    print(f"- Prediksi CNN: {sentiment_cnn}")


=== HASIL PREDIKSI SENTIMEN ===
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 122ms/step

Review: Aplikasi ini sangat bagus, saya sangat puas!
- Prediksi SVM: neutral
- Prediksi Random Forest: positive
- Prediksi CNN: positive
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step

Review: Sangat mengecewakan, banyak bug dan sering crash.
- Prediksi SVM: neutral
- Prediksi Random Forest: negative
- Prediksi CNN: negative
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step

Review: Aplikasi ini lumayan, tidak buruk tapi juga tidak terlalu bagus.
- Prediksi SVM: neutral
- Prediksi Random Forest: positive
- Prediksi CNN: negative


Hasil prediksi sentimen menunjukkan bahwa model **SVM**, **Random Forest**, dan **CNN** memberikan hasil yang bervariasi pada beberapa review. Model **SVM cenderung lebih sering memprediksi "neutral"**, sedangkan **Random Forest dan CNN lebih konsisten dalam mendeteksi sentimen positif/negatif**. Contoh terakhir menunjukkan adanya perbedaan interpretasi antara model, di mana **Random Forest memprediksi "positive"** sementara **CNN memprediksi "negative"**, menandakan perbedaan sensitivitas dalam menangkap nuansa sentimen.

## **Kesimpulan**

Dari hasil analisis sentimen review aplikasi **LINE**, tiga model yang digunakan—**SVM, Random Forest, dan CNN**—menunjukkan performa yang sangat baik dengan akurasi tinggi pada dataset uji. **Random Forest dan CNN mencapai akurasi di atas 98%**, sementara **SVM menunjukkan performa sedikit lebih tinggi dengan akurasi 99.39%** pada skema pembagian data 70/30. Namun, dalam pengujian pada review individu, terdapat perbedaan prediksi antar model, terutama dalam menangkap sentimen netral. **SVM cenderung lebih sering memprediksi "neutral"**, sedangkan **Random Forest dan CNN lebih jelas dalam membedakan sentimen positif dan negatif**. Dengan akurasi yang sangat tinggi, model ini dapat digunakan untuk analisis sentimen otomatis, tetapi tetap perlu dievaluasi lebih lanjut untuk menangani ambiguitas dalam teks.